In [1]:
import os
while not os.path.exists('METHODOLOGY.md') and os.getcwd() != '/': os.chdir('..')
assert os.path.exists('METHODOLOGY.md'), 'run from inside the Nofit_LRT_Extension repository'

# Step 39 — TAZ-level LRT capture on the 174 corridor TAZs, on the TAZ-level skims of step 44 (`LRT_capture_TAZ.ipynb`, task C3 / A2 / E6)

Step 31 (§6ac) runs the capture on the 25 V2 areas: an incremental logit pivoted on each area
pair's observed 2022 transit share, with the LRT's share of the transit nest set by the
generalized-cost gap between the LRT and the bus. Because the logit is nonlinear in access
time, averaging the walk over an area before applying it biases the result: a TAZ 300 m from
a station and one 2 km away are not the area's average twice over. The hand-over's task C3
asked for the same pivot on the **174 V2 TAZs**, with the walks at TAZ level; step 44 (§6an)
has since provided car and transit skims at TAZ level, and this step runs the capture on them.

**TAZ-pair generalized costs.** *Bus (the transit nest's existing alternative)*: step 44's
TAZ-level transit skim — the best bus / Metronit path from the TAZ centroid, with its own walk,
wait and transfers (`BUS_TAZ_SOURCE = 'los'`, the default). The hand-over's construction is
kept as the alternative (`'area_plus_walk'`): the area pair's bus in-vehicle time, wait and
transfers with 2 × the TAZ's own stop walks in place of the area's. *LRT, per scenario*: the
area pair's LRT skim of step 31 (in-vehicle time, wait, transfer penalties, the feeder
composition and gateway stations it recorded) with the two side walks replaced at TAZ level —
the TAZ's walk to its nearest station where that end is on the line (`lrt_access_taz_v2.csv`),
the TAZ's walk to its nearest bus stop where that end reaches the LRT by a feeder. *Car*:
step 44's TAZ skim, carried for the record (the pivot does not use the car's cost level,
caveat 25). The pivot's formulas, cases and constants are step 31's, copied, not imported.

**The pivot per TAZ pair.** Observed 2022 car and transit trips of the TAZ pair
(`Output/final_2022/`), transit share Empirical-Bayes-smoothed towards the **area pair's**
share with k = 20 trips; `P_LRT|T`, the nest logsum gain and the car → transit shift as in
step 31; central case and the four λ / premium cases; the all-underground, all-ground and
design regimes. Results are summed back to areas and loaded on the trunk links by the area
pair's gateways, and the capture rate is read by the origin TAZ's station-access band.

**Checks.** With every TAZ given its area's walks and the hand-over's bus construction, the
TAZ run should reproduce the area result closely (the regression test of the construction);
with the LRT removed the 2022 flows are returned exactly.

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BLUE, ORANGE, AQUA, PURPLE, INK, INK2, MUTED, GRID, AXIS = '#2a78d6', '#eb6834', '#1baf7a', '#7b5bd6', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
BUS_TAZ_SOURCE = os.environ.get('BUS_TAZ_SOURCE', 'los'); assert BUS_TAZ_SOURCE in ('los', 'area_plus_walk')
OUT = 'Output/skims/taz' if BUS_TAZ_SOURCE == 'los' else 'Output/skims/taz/bus_area_plus_walk'; os.makedirs(OUT, exist_ok=True); FIG = 'Output/figures'
W_WALK, W_WAIT, TRANSFER_PEN, WALK_SPEED, DETOUR, K_EB = 2.0, 2.0, 8.0, 80.0, 1.3, 20.0
LAM_CASES = {'central (λ 0.03, λ_T 0.06, premium 5)': (0.03, 0.06, 5.0), 'low λ (0.02 / 0.03, premium 5)': (0.02, 0.03, 5.0), 'high λ (0.05 / 0.10, premium 5)': (0.05, 0.10, 5.0),
             'no premium (λ central, premium 0)': (0.03, 0.06, 0.0), 'premium 10 (λ central)': (0.03, 0.06, 10.0)}
CENTRAL = 'central (λ 0.03, λ_T 0.06, premium 5)'
LRT_SCEN = ['lrt_all_underground', 'lrt_all_ground', 'lrt_design_50kmh']
LABEL = {'lrt_all_underground': 'LRT all underground', 'lrt_all_ground': 'LRT all ground', 'lrt_design_50kmh': 'LRT design 50 km/h'}
SK_DIR = 'Output/skims'
print(f"BUS_TAZ_SOURCE = '{BUS_TAZ_SOURCE}' -- writing to {OUT}")

xl = pd.ExcelFile('Input/Corridor_TAZ_Agg_V2.xlsx'); areas = xl.parse('AreaCodes').set_index('AggCode'); key = xl.parse('TazAgg')
AREAS = list(areas.index); names = areas['AggAreaName']; A = len(AREAS); apos = {a: i for i, a in enumerate(AREAS)}
TAZS = sorted(key['TAZ'].astype(int)); NT = len(TAZS); tpos = {t: i for i, t in enumerate(TAZS)}
area_of = key.set_index('TAZ')['AggCode'].reindex(TAZS).astype(int); ta = area_of.values; tai = np.array([apos[a] for a in ta])
zon = pd.read_csv('Input/Zonal_2020.csv', encoding='windows-1255').set_index('TAZ_ID')
pop = zon['POPULATION'].reindex(TAZS).fillna(0).values; emp = zon['EMPL_TOT'].reindex(TAZS).fillna(0).values
lrt_acc_t = pd.read_csv('Output/gc/lrt_access_taz_v2.csv').set_index('TAZ').reindex(TAZS)
lrt_acc_a = pd.read_csv('Output/gc/lrt_access_area_v2.csv', index_col=0).reindex(AREAS)
ON_LINE = [a for a in AREAS if bool(lrt_acc_a.loc[a, 'served_by_hf_lrt_3'])]; on_line_t = np.isin(ta, ON_LINE)
los = pd.read_csv('Output/gtfs/bus_los_taz.csv').set_index('TAZ')
bus_walk_t = (los['bus_nearest_stop_m'].reindex(TAZS) * DETOUR / WALK_SPEED).values
def wavg(v, w): return np.average(v, weights=w) if w.sum() > 0 else v.mean()
accb = pd.DataFrame({'access': [wavg(bus_walk_t[ta == a], pop[ta == a]) for a in AREAS], 'egress': [wavg(bus_walk_t[ta == a], emp[ta == a]) for a in AREAS]}, index=AREAS)
lrt_walk_t = lrt_acc_t['walk_min'].values; lrt_dist_t = lrt_acc_t['dist_m'].values
def rd(f, obj=False):
    m = pd.read_csv(f, index_col=0, dtype=object if obj else None); m.columns = m.columns.astype(int); m = m.reindex(index=AREAS, columns=AREAS); return m if obj else m.astype(float)
SKA = {'bus': {c: rd(f'{SK_DIR}/skim_bus_{c}.csv') for c in ('gc', 'ivt', 'walk', 'wait', 'transfers')}}
for sc in LRT_SCEN:
    SKA[sc] = {c: rd(f'{SK_DIR}/skim_{sc}_{c}.csv') for c in ('gc', 'ivt', 'walk', 'wait', 'transfers')}
    SKA[sc].update({c: rd(f'{SK_DIR}/skim_{sc}_{c}.csv', True).fillna('') for c in ('gateway_o', 'gateway_d', 'legs')})
# side walks of the area skims: the LRT station walk where that end is on the line, the bus stop walk where it reaches the LRT by a feeder
side_o_area = np.where(np.isin(AREAS, ON_LINE), lrt_acc_a['access_walk_min_pop'].values, accb['access'].values)
side_d_area = np.where(np.isin(AREAS, ON_LINE), lrt_acc_a['egress_walk_min_emp'].values, accb['egress'].values)
for sc in LRT_SCEN:
    recon = side_o_area[:, None] + side_d_area[None, :]; w = SKA[sc]['walk'].values; ok = np.isfinite(w) & ~np.eye(A, dtype=bool)
    assert np.nanmax(np.abs(recon[ok] - w[ok])) < 0.05, f'{sc}: the LRT walk of step 31 is not the sum of the two side walks'
side_o_taz = np.where(on_line_t, lrt_walk_t, bus_walk_t); side_d_taz = side_o_taz.copy()
# the 2022 TAZ layers and the area flows
def rdm(path, idx):
    m = pd.read_csv(path, index_col=0); m.columns = m.columns.astype(int); m.index = m.index.astype(int); return m.reindex(index=idx, columns=idx).fillna(0.0).values
T_car = rdm('Output/final_2022/car_2022_taz.csv', TAZS); T_tr = rdm('Output/final_2022/transit_2022_taz.csv', TAZS)
# the universe is the TAZ pairs in DIFFERENT areas: the area diagonal is not skimmed for the LRT at either level (step 31 excludes it), so the
# intra-area TAZ pairs are left out here too, and the totals and shares below are on the same footing as step 31's
OFF = (ta[:, None] != ta[None, :]); T_car *= OFF; T_tr *= OFF
F_car = rdm('Output/corridor_v2/car_2022_area_v2.csv', AREAS); F_tr = rdm('Output/corridor_v2/transit_2022_area_v2.csv', AREAS); OFFA = ~np.eye(A, dtype=bool); F_car *= OFFA; F_tr *= OFFA
agg_car = np.zeros((A, A)); agg_tr = np.zeros((A, A)); np.add.at(agg_car, (tai[:, None].repeat(NT, 1), tai[None, :].repeat(NT, 0)), T_car); np.add.at(agg_tr, (tai[:, None].repeat(NT, 1), tai[None, :].repeat(NT, 0)), T_tr)
print(f'{NT} TAZs in {A} areas, {int(on_line_t.sum())} on the line; 2022 AM trips on the inter-area TAZ pairs: car {T_car.sum():,.0f}, transit {T_tr.sum():,.0f} (area flows of step 24: {F_car.sum():,.0f} / {F_tr.sum():,.0f}); '
      f'the TAZ layers summed to area pairs reproduce the area flows to {np.abs(agg_car - F_car).max():.2f} (car) and {np.abs(agg_tr - F_tr).max():.2f} (transit) trips at the worst pair')

BUS_TAZ_SOURCE = 'los' -- writing to Output/skims/taz


174 TAZs in 25 areas, 64 on the line; 2022 AM trips on the inter-area TAZ pairs: car 56,445, transit 14,133 (area flows of step 24: 56,445 / 14,133); the TAZ layers summed to area pairs reproduce the area flows to 0.00 (car) and 0.00 (transit) trips at the worst pair


## 1. The TAZ-pair generalized costs

In [3]:
def taz_from_area(M): return M[np.ix_(tai, tai)]
# bus: step 44's TAZ-level best path, or the area components with the TAZ's stop walks
los_t = pd.read_csv('Output/los/transit_los_taz.csv.gz'); los_t = los_t[los_t['o'].isin(TAZS) & los_t['d'].isin(TAZS)]
bus_gc_los = np.full((NT, NT), np.nan); bus_gc_los[los_t['o'].map(tpos).values, los_t['d'].map(tpos).values] = los_t['transit_gc_min'].values
bus_gc_apw = taz_from_area(SKA['bus']['gc'].values) - W_WALK * (taz_from_area(accb['access'].values[:, None] + accb['egress'].values[None, :]) - (bus_walk_t[:, None] + bus_walk_t[None, :]))
bus_gc = np.where(np.isfinite(bus_gc_los), bus_gc_los, bus_gc_apw) if BUS_TAZ_SOURCE == 'los' else bus_gc_apw
car_t = pd.read_csv('Output/los/car_los_taz.csv.gz'); car_t = car_t[car_t['o'].isin(TAZS) & car_t['d'].isin(TAZS)]
car_gc = np.full((NT, NT), np.nan); car_gc[car_t['o'].map(tpos).values, car_t['d'].map(tpos).values] = car_t['car_gc_min'].values
# LRT: the area pair's skim with the two side walks at TAZ level
LRT = {}
for sc in LRT_SCEN:
    gc_a = taz_from_area(SKA[sc]['gc'].values); walk_a = taz_from_area(side_o_area[:, None] + side_d_area[None, :]); walk_t = side_o_taz[:, None] + side_d_taz[None, :]
    LRT[sc] = {'gc': gc_a - W_WALK * walk_a + W_WALK * walk_t, 'walk': walk_t, 'ivt': taz_from_area(SKA[sc]['ivt'].values), 'wait': taz_from_area(SKA[sc]['wait'].values), 'transfers': taz_from_area(SKA[sc]['transfers'].values),
               'gateway_o': SKA[sc]['gateway_o'].values[np.ix_(tai, tai)], 'gateway_d': SKA[sc]['gateway_d'].values[np.ix_(tai, tai)]}
n_pair = T_car + T_tr; m = OFF & (n_pair > 0)
wt = T_tr * OFF
def tw(M, w=wt): ok = np.isfinite(M) & (w > 0); return np.average(M[ok], weights=w[ok])
trunk_t = on_line_t[:, None] & on_line_t[None, :] & OFF
print(f"TAZ pairs with 2022 trips: {int(m.sum()):,} of {int(OFF.sum()):,}; bus GC source '{BUS_TAZ_SOURCE}' ({'step 44 path for ' + format(int((np.isfinite(bus_gc_los) & m).sum()), ',') + ' pairs, area construction for the rest' if BUS_TAZ_SOURCE == 'los' else 'area components + TAZ walks'})")
print('transit-trip-weighted generalized minutes on the TAZ pairs (trunk TAZ pairs / all): ' + '; '.join([f"bus {tw(np.where(trunk_t, bus_gc, np.nan)):.1f} / {tw(bus_gc):.1f}"] + [f"{LABEL[sc]} {tw(np.where(trunk_t, LRT[sc]['gc'], np.nan)):.1f} / {tw(LRT[sc]['gc']):.1f}" for sc in LRT_SCEN]) +
      f"; car {tw(np.where(trunk_t, car_gc, np.nan), T_car * OFF):.1f} / {tw(car_gc, T_car * OFF):.1f}")
print(f"the same on the area skims (area pairs, transit-trip-weighted): bus {tw(np.where(np.isin(AREAS, ON_LINE)[:, None] & np.isin(AREAS, ON_LINE)[None, :] & OFFA, SKA['bus']['gc'].values, np.nan), F_tr):.1f} / {tw(SKA['bus']['gc'].values, F_tr):.1f}; "
      + '; '.join(f"{LABEL[sc]} {tw(np.where(np.isin(AREAS, ON_LINE)[:, None] & np.isin(AREAS, ON_LINE)[None, :] & OFFA, SKA[sc]['gc'].values, np.nan), F_tr):.1f} / {tw(SKA[sc]['gc'].values, F_tr):.1f}" for sc in LRT_SCEN))

TAZ pairs with 2022 trips: 23,488 of 27,836; bus GC source 'los' (step 44 path for 23,488 pairs, area construction for the rest)
transit-trip-weighted generalized minutes on the TAZ pairs (trunk TAZ pairs / all): bus 41.7 / 50.6; LRT all underground 43.2 / 59.4; LRT all ground 51.8 / 65.1; LRT design 50 km/h 38.2 / 55.7; car 13.8 / 16.3
the same on the area skims (area pairs, transit-trip-weighted): bus 30.7 / 37.2; LRT all underground 46.6 / 62.5; LRT all ground 55.2 / 68.3; LRT design 50 km/h 41.6 / 58.8


## 2. The pivot per TAZ pair

In [4]:
S_area = np.where(F_car + F_tr > 0, F_tr / np.where(F_car + F_tr > 0, F_car + F_tr, 1), np.nan); p0 = F_tr.sum() / (F_car + F_tr).sum()
S_prior = np.nan_to_num(taz_from_area(S_area), nan=p0)
S_raw = np.where(n_pair > 0, T_tr / np.where(n_pair > 0, n_pair, 1), np.nan)
S_piv = np.where(n_pair > 0, (n_pair * np.nan_to_num(S_raw) + K_EB * S_prior) / (n_pair + K_EB), np.nan)
def pivot(bus_gc_, lrt_gc_, lam, lt, prem):
    dL = lrt_gc_ - prem - bus_gc_
    avail = np.isfinite(dL) & OFF & (n_pair > 0)
    with np.errstate(over='ignore', invalid='ignore'):
        P_L = np.where(avail, 1 / (1 + np.exp(lt * np.nan_to_num(dL))), 0.0)
        delta = np.where(avail, -(1 / lt) * np.log1p(np.exp(-lt * np.nan_to_num(dL))), 0.0)
        S_new = np.where(avail, S_piv * np.exp(-lam * delta) / (S_piv * np.exp(-lam * delta) + 1 - S_piv), S_piv)
    growth = np.where(avail, np.nan_to_num(S_new) / np.where(np.nan_to_num(S_piv) > 0, S_piv, 1), 1.0)
    T_tr_new = np.minimum(T_tr * growth, n_pair); T_car_new = n_pair - T_tr_new
    T_lrt = T_tr_new * P_L; from_bus = T_tr * P_L
    return {'P_LRT|T': P_L, 'T_lrt': T_lrt, 'T_bus': T_tr_new - T_lrt, 'T_car': T_car_new, 'T_tr_new': T_tr_new, 'from_bus': from_bus, 'from_car': T_lrt - from_bus, 'avail': avail}
RES = {}; rows = []
for sc in LRT_SCEN:
    for lab, (lam, lt, prem) in LAM_CASES.items():
        r = pivot(bus_gc, LRT[sc]['gc'], lam, lt, prem); RES[(sc, lab)] = r
        rows.append({'scenario': LABEL[sc], 'case': lab, 'λ': lam, 'λ_T': lt, 'LRT premium': prem, 'LRT trips 06–09': round(float(r['T_lrt'].sum())), 'of which from bus': round(float(r['from_bus'].sum())), 'of which from car': round(float(r['from_car'].sum())),
                     'bus trips after': round(float(r['T_bus'].sum())), 'car trips after': round(float(r['T_car'].sum())), 'transit share before': round(float(T_tr.sum() / n_pair.sum()), 4), 'transit share after': round(float(r['T_tr_new'].sum() / n_pair.sum()), 4),
                     'LRT share of transit': round(float(r['T_lrt'].sum() / r['T_tr_new'].sum()), 3), 'LRT trips on trunk pairs': round(float(r['T_lrt'][trunk_t].sum())), 'trunk pairs P_LRT|T (trip-wtd)': round(float(np.average(r['P_LRT|T'][trunk_t], weights=np.maximum(T_tr[trunk_t], 1e-9))), 3),
                     'TAZ pairs with LRT available': int(r['avail'].sum())})
scen = pd.DataFrame(rows); scen.to_csv(f'{OUT}/lrt_capture_scenarios_taz.csv', index=False); print(scen.drop(columns=['λ', 'λ_T', 'LRT premium']).to_string(index=False))
# no-build check: with the LRT removed (its cost infinite) the 2022 flows come back exactly
r0 = pivot(bus_gc, np.full((NT, NT), np.inf), 0.03, 0.06, 5.0); assert np.allclose(r0['T_tr_new'], T_tr) and np.allclose(r0['T_car'], T_car) and r0['T_lrt'].sum() == 0, 'no-build flows not returned'
print('no-build check passed: with the LRT removed the 2022 TAZ flows are returned exactly')
for sc in LRT_SCEN:
    r = RES[(sc, CENTRAL)]
    pd.DataFrame(r['T_lrt'], index=TAZS, columns=TAZS).to_csv(f'{OUT}/lrt_trips_2022_taz_{sc}_central.csv', float_format='%.2f')
    pd.DataFrame(r['P_LRT|T'], index=TAZS, columns=TAZS).to_csv(f'{OUT}/lrt_share_of_transit_taz_{sc}_central.csv', float_format='%.3f')
pd.DataFrame(S_piv, index=TAZS, columns=TAZS).to_csv(f'{OUT}/transit_share_2022_pivot_taz.csv', float_format='%.4f')
pd.DataFrame({'TAZ': TAZS, 'area': ta, 'on_line': on_line_t, 'pop': pop, 'emp': emp, 'lrt_station': lrt_acc_t['nearest_station'].values, 'lrt_walk_dist_m': lrt_dist_t, 'lrt_walk_min': lrt_walk_t, 'bus_stop_walk_min': bus_walk_t}).to_csv(f'{OUT}/taz_access_table.csv', index=False, float_format='%.2f')

           scenario                                  case  LRT trips 06–09  of which from bus  of which from car  bus trips after  car trips after  transit share before  transit share after  LRT share of transit  LRT trips on trunk pairs  trunk pairs P_LRT|T (trip-wtd)  TAZ pairs with LRT available
LRT all underground central (λ 0.03, λ_T 0.06, premium 5)             6963               6496                466             8082            55534                0.2003               0.2132                 0.463                      2695                           0.547                         23488
LRT all underground        low λ (0.02 / 0.03, premium 5)             7341               6708                634             8172            55065                0.2003               0.2198                 0.473                      2635                           0.525                         23488
LRT all underground       high λ (0.05 / 0.10, premium 5)             6981               6379       

## 3. Back to the areas, the trunk links and the station areas; the capture by access band

In [5]:
area_scen = pd.read_csv(f'{SK_DIR}/lrt_capture_scenarios.csv')
def to_area(M): out = np.zeros((A, A)); np.add.at(out, (tai[:, None].repeat(NT, 1), tai[None, :].repeat(NT, 0)), np.nan_to_num(M)); return out
cmp_rows = []
for sc in LRT_SCEN:
    for lab in LAM_CASES:
        r = RES[(sc, lab)]; a = area_scen[(area_scen['scenario'] == LABEL[sc]) & (area_scen['case'] == lab)].iloc[0]
        cmp_rows.append({'scenario': LABEL[sc], 'case': lab, 'LRT trips — area level (step 31)': a['LRT trips 06–09'], 'LRT trips — TAZ level': round(float(r['T_lrt'].sum())), 'ratio TAZ / area': round(float(r['T_lrt'].sum()) / a['LRT trips 06–09'], 3),
                         'from bus — area': a['of which from bus'], 'from bus — TAZ': round(float(r['from_bus'].sum())), 'from car — area': a['of which from car'], 'from car — TAZ': round(float(r['from_car'].sum())),
                         'trunk pairs — area': a['LRT trips on trunk pairs'], 'trunk pairs — TAZ': round(float(r['T_lrt'][trunk_t].sum())), 'P_LRT|T trunk — area': a['trunk pairs P_LRT|T (trip-wtd)'], 'P_LRT|T trunk — TAZ': round(float(np.average(r['P_LRT|T'][trunk_t], weights=np.maximum(T_tr[trunk_t], 1e-9))), 3)})
cmp_ = pd.DataFrame(cmp_rows); cmp_.to_csv(f'{OUT}/taz_vs_area_capture_comparison.csv', index=False); print(cmp_.to_string(index=False))
# by origin area, central case: TAZ-level sums beside the area-level matrix
by_area = []
for sc in LRT_SCEN:
    Ta = to_area(RES[(sc, CENTRAL)]['T_lrt']); Aa = rd(f'{SK_DIR}/lrt_trips_2022_{sc}_central.csv').fillna(0).values
    pd.DataFrame(Ta, index=AREAS, columns=AREAS).to_csv(f'{OUT}/lrt_trips_2022_area_from_taz_{sc}_central.csv', float_format='%.1f')
    by_area.append(pd.DataFrame({'area': AREAS, 'name': names.values, 'scenario': LABEL[sc], 'lrt_origins_area_level': Aa.sum(1).round(0), 'lrt_origins_taz_level': Ta.sum(1).round(0), 'lrt_destinations_area_level': Aa.sum(0).round(0), 'lrt_destinations_taz_level': Ta.sum(0).round(0)}))
by_area = pd.concat(by_area); by_area.to_csv(f'{OUT}/lrt_trips_by_area_taz_vs_area.csv', index=False)
# trunk links by the area pair's gateways (step 31's rule) and boardings by gateway station area / by nearest station
lf = pd.read_csv('Output/corridor_v2/corridor_v2_network_link_flows.csv')
trunk_links = lf[lf['segment'] == 'trunk'][['link', 'from_code', 'to_code', 'from_area', 'to_area']].drop_duplicates().sort_values('link')
trunk_links = trunk_links[trunk_links['to_code'].isin(ON_LINE) & trunk_links['from_code'].isin(ON_LINE)].reset_index(drop=True); pos = {a: i for i, a in enumerate(ON_LINE)}
def load_links(Ta, g_o, g_d):
    L = {(l, d): 0.0 for l in trunk_links['link'] for d in ('up', 'down')}
    for i, o in enumerate(AREAS):
        for j, d in enumerate(AREAS):
            t = Ta[i, j]
            if t <= 0 or g_o.loc[o, d] == '': continue
            a, b = pos[int(float(g_o.loc[o, d]))], pos[int(float(g_d.loc[o, d]))]
            if a == b: continue
            for k in range(min(a, b), max(a, b)): L[(trunk_links.loc[k, 'link'], 'up' if b > a else 'down')] += t
    return L
prof = pd.DataFrame([{'link': r['link'], 'from_area': r['from_area'], 'to_area': r['to_area'], 'direction': d} for _, r in trunk_links.iterrows() for d in ('up', 'down')])
area_prof = pd.read_csv(f'{SK_DIR}/trunk_link_flows_bus_vs_lrt.csv')
tags = {'lrt_all_underground': 'lrt_ug', 'lrt_all_ground': 'lrt_ground', 'lrt_design_50kmh': 'lrt_d50'}
for sc in LRT_SCEN:
    L = load_links(to_area(RES[(sc, CENTRAL)]['T_lrt']), SKA[sc]['gateway_o'], SKA[sc]['gateway_d'])
    prof[f'{tags[sc]}_3h_taz'] = [round(L[(l, d)], 1) for l, d in zip(prof['link'], prof['direction'])]
    prof[f'{tags[sc]}_3h_area'] = prof.merge(area_prof[['link', 'direction', f'{tags[sc]}_3h']], on=['link', 'direction'], how='left')[f'{tags[sc]}_3h'].values
prof.to_csv(f'{OUT}/trunk_link_flows_taz.csv', index=False)
busiest = prof.loc[prof['lrt_ug_3h_taz'].idxmax()]; busiest_a = prof.loc[prof['lrt_ug_3h_area'].idxmax()]
print(f"busiest trunk link, underground central: TAZ level {busiest['from_area']} → {busiest['to_area']} ({busiest['direction']}) {busiest['lrt_ug_3h_taz']:,.0f} in three hours; area level {busiest_a['from_area']} → {busiest_a['to_area']} ({busiest_a['direction']}) {busiest_a['lrt_ug_3h_area']:,.0f}")
bo = []
for sc in LRT_SCEN:
    Ta = to_area(RES[(sc, CENTRAL)]['T_lrt']); g_o = SKA[sc]['gateway_o']; s = pd.Series(0.0, index=ON_LINE)
    for i, o in enumerate(AREAS):
        for j, d in enumerate(AREAS):
            if Ta[i, j] > 0 and g_o.loc[o, d] != '': s[int(float(g_o.loc[o, d]))] += Ta[i, j]
    bo.append(s.rename(LABEL[sc]))
board = pd.concat(bo, axis=1); board.insert(0, 'area', names.reindex(ON_LINE).values)
board_a = pd.read_csv(f'{SK_DIR}/lrt_boardings_by_station_area_central.csv', index_col=0)
for sc in LRT_SCEN: board[f'{LABEL[sc]} — area level'] = board_a[LABEL[sc]].reindex(ON_LINE).values
board.round(0).to_csv(f'{OUT}/lrt_boardings_by_station_area_taz.csv')
# by nearest station, for the trips boarding at an on-line origin TAZ (the off-line origins board at their gateway area)
st_rows = []
for sc in LRT_SCEN:
    T = RES[(sc, CENTRAL)]['T_lrt']; s = pd.Series(T[on_line_t].sum(1), index=np.array(TAZS)[on_line_t]).groupby(lrt_acc_t['nearest_station'].reindex(np.array(TAZS)[on_line_t]).values).sum()
    st_rows.append(s.rename(LABEL[sc]))
pd.concat(st_rows, axis=1).round(0).to_csv(f'{OUT}/lrt_boardings_by_station_taz.csv')
# capture by the origin TAZ's station-access band (on-line origins): LRT trips, transit trips, the capture rate
BANDS = [(0, 500), (500, 1000), (1000, 1500), (1500, 1e9)]
band_rows = []
for sc in LRT_SCEN:
    r = RES[(sc, CENTRAL)]
    for lo, hi in BANDS:
        sel = on_line_t & (lrt_dist_t >= lo) & (lrt_dist_t < hi)
        band_rows.append({'scenario': LABEL[sc], 'origin station access (walking m)': f'{lo:.0f}–{hi:.0f}' if hi < 1e9 else f'> {lo:.0f}', 'TAZs': int(sel.sum()), 'population': pop[sel].sum(),
                          'transit trips 2022': T_tr[sel].sum(), 'LRT trips': r['T_lrt'][sel].sum(), 'capture rate (LRT / transit after)': r['T_lrt'][sel].sum() / max(r['T_tr_new'][sel].sum(), 1e-9), 'from car': r['from_car'][sel].sum()})
    sel = ~on_line_t
    band_rows.append({'scenario': LABEL[sc], 'origin station access (walking m)': 'off the line (feeder)', 'TAZs': int(sel.sum()), 'population': pop[sel].sum(), 'transit trips 2022': T_tr[sel].sum(), 'LRT trips': r['T_lrt'][sel].sum(),
                      'capture rate (LRT / transit after)': r['T_lrt'][sel].sum() / max(r['T_tr_new'][sel].sum(), 1e-9), 'from car': r['from_car'][sel].sum()})
bands = pd.DataFrame(band_rows); bands.to_csv(f'{OUT}/lrt_capture_by_access_band.csv', index=False, float_format='%.3f'); print(bands.round(2).to_string(index=False))

           scenario                                  case  LRT trips — area level (step 31)  LRT trips — TAZ level  ratio TAZ / area  from bus — area  from bus — TAZ  from car — area  from car — TAZ  trunk pairs — area  trunk pairs — TAZ  P_LRT|T trunk — area  P_LRT|T trunk — TAZ
LRT all underground central (λ 0.03, λ_T 0.06, premium 5)                              4254                   6963             1.637             3841            6496              414             466                1767               2695                 0.345                0.547
LRT all underground        low λ (0.02 / 0.03, premium 5)                              6031                   7341             1.217             5113            6708              917             634                2315               2635                 0.419                0.525
LRT all underground       high λ (0.05 / 0.10, premium 5)                              3166                   6981             2.205             2890        

busiest trunk link, underground central: TAZ level Namal-Giborim → Hamifrats (down) 2,662 in three hours; area level Namal-Giborim → Hamifrats (down) 1,673
           scenario origin station access (walking m)  TAZs  population  transit trips 2022  LRT trips  capture rate (LRT / transit after)  from car
LRT all underground                             0–500    27       32640             2796.48    1708.81                                0.57    107.99
LRT all underground                          500–1000    28       36585             2137.62    1266.05                                0.56     74.08
LRT all underground                         1000–1500     3        5282              218.91     152.14                                0.59     25.17
LRT all underground                            > 1500     6        3663              124.47      61.21                                0.44      6.50
LRT all underground             off the line (feeder)   110      414172             8855.93    3774

## 4. The regression check: the TAZ construction with the area's own walks

In [6]:
# every TAZ given its area's walks, the bus from the area components: the only remaining difference from step 31 is the pivot's own nonlinearity across the TAZ pairs of an area pair
bus_gc_chk = taz_from_area(SKA['bus']['gc'].values); chk_rows = []
for sc in LRT_SCEN:
    lrt_gc_chk = taz_from_area(SKA[sc]['gc'].values); r = pivot(bus_gc_chk, lrt_gc_chk, *LAM_CASES[CENTRAL])
    a = area_scen[(area_scen['scenario'] == LABEL[sc]) & (area_scen['case'] == CENTRAL)].iloc[0]
    chk_rows.append({'scenario': LABEL[sc], 'area level (step 31)': a['LRT trips 06–09'], 'TAZ level with the area skims': round(float(r['T_lrt'].sum())), 'ratio': round(float(r['T_lrt'].sum()) / a['LRT trips 06–09'], 4),
                     'from bus — area': a['of which from bus'], 'from bus — TAZ': round(float(r['from_bus'].sum())), 'from car — area': a['of which from car'], 'from car — TAZ': round(float(r['from_car'].sum()))})
chk = pd.DataFrame(chk_rows); chk.to_csv(f'{OUT}/regression_check_area_skims_on_taz.csv', index=False); print(chk.to_string(index=False))
print(f'the trips drawn from the bus reproduce the area result exactly (same P_LRT|T on every TAZ pair of an area pair); the trips induced from the car fall, because the pivot now acts on each TAZ pair\'s own '
      f'EB-smoothed share and the TAZ-pair shares are more dispersed than the area pair\'s (S(1-S) is smaller) — the construction reproduces the area total to within {100 * (chk["ratio"] - 1).abs().max():.1f} %, the gap being that induced component')

           scenario  area level (step 31)  TAZ level with the area skims  ratio  from bus — area  from bus — TAZ  from car — area  from car — TAZ
LRT all underground                  4254                           3970 0.9333             3841            3841              414             130
     LRT all ground                  3206                           3050 0.9513             2972            2972              234              77
 LRT design 50 km/h                  5095                           4668 0.9161             4481            4481              614             187
the trips drawn from the bus reproduce the area result exactly (same P_LRT|T on every TAZ pair of an area pair); the trips induced from the car fall, because the pivot now acts on each TAZ pair's own EB-smoothed share and the TAZ-pair shares are more dispersed than the area pair's (S(1-S) is smaller) — the construction reproduces the area total to within 8.4 %, the gap being that induced component


In [7]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5.2))
ba = by_area[by_area['scenario'] == 'LRT all underground'].sort_values('lrt_origins_area_level', ascending=False)
xs = np.arange(len(ba)); ax[0].bar(xs - .2, ba['lrt_origins_area_level'], .4, color=MUTED, label='area level (step 31)'); ax[0].bar(xs + .2, ba['lrt_origins_taz_level'], .4, color=BLUE, label=f'TAZ level (step 39, bus = {BUS_TAZ_SOURCE})')
ax[0].set_xticks(xs); ax[0].set_xticklabels(ba['name'], rotation=70, ha='right', fontsize=7); ax[0].set_ylabel('LRT trips 06:00–09:00 by origin area'); ax[0].set_title('LRT all underground, central case: origins by area', fontsize=10); ax[0].legend(frameon=False, fontsize=8)
bb = bands[bands['scenario'] == 'LRT all underground']; xs = np.arange(len(bb))
ax[1].bar(xs, bb['capture rate (LRT / transit after)'], color=AQUA, width=.6)
for i, (v, n) in enumerate(zip(bb['capture rate (LRT / transit after)'], bb['LRT trips'])): ax[1].text(i, v + .01, f'{v:.2f}\n({n:,.0f} trips)', ha='center', fontsize=8)
ax[1].set_xticks(xs); ax[1].set_xticklabels(bb['origin station access (walking m)'], fontsize=8); ax[1].set_ylabel('LRT share of transit after (origin TAZs)'); ax[1].set_title('Capture rate by the origin TAZ\'s station access, underground central', fontsize=10)
for a_ in ax:
    for sp in ('top', 'right'): a_.spines[sp].set_visible(False)
    a_.grid(axis='y', color=GRID, lw=.6)
plt.tight_layout(); plt.savefig(f"{FIG}/lrt_capture_taz_vs_area{'' if BUS_TAZ_SOURCE == 'los' else '_bus_area_plus_walk'}.png", dpi=150); plt.close(); print('figure written')

figure written


## 5. Results, outputs, limits

**The universe.** 174 TAZs, 64 on the line; 27,836 inter-area TAZ pairs, 23,488 with 2022
trips (car 56,445, transit 14,133 — the area flows exactly, since the intra-area TAZ pairs are
left out as the area diagonal is). On the trunk TAZ pairs, transit-trip-weighted, the bus costs
**41.7** generalized minutes on step 44's path against **30.7** in the area skim; the LRT
**43.2** (underground) against 46.6, the TAZ station walks being shorter than the area's
population-weighted average for the transit riders' TAZs.

**The capture on the step-44 bus skim** (`'los'`, the default). Underground central
**6,963 LRT trips 06:00–09:00** (6,496 from the bus, 466 from the car) against **4,254** at the
area level — × 1.64; ground 5,811 vs 3,206; design regime 7,726 vs 5,095; `P_LRT|T` on the trunk
pairs 0.55 vs 0.35; the busiest link (Namal-Giborim → Hamifrats, down) 2,662 against 1,673;
boardings led by S12 (418), S10 (358), S02 (250). The capture rate by the origin TAZ's station
access is 0.57 / 0.56 / 0.59 / 0.44 for < 500 / 500–1,000 / 1,000–1,500 / > 1,500 m and 0.40
for the off-line origins — no gradient, because the bus's own access cost grows with the same
distance. **Read with care**: the rise over the area result is the caveat-26 effect, the bus
alternative priced at what a TAZ traveller faces while the LRT keeps the area skim's
in-vehicle time and, for the 110 off-line TAZs, the area skim's *optimistic* bus feeder leg —
the feeder pairs' capture rate (0.40) is where that asymmetry bites.

**The capture on the hand-over's construction** (`'area_plus_walk'`: the area bus components
with the TAZ's stop walks). Underground central **4,226** against 4,254 (0.99); ground 3,280
(1.02); design 4,924 (0.97); across the cases 0.91–1.12. The access gradient the task expected
appears: 0.42 / 0.34 / 0.32 / 0.23 by band and 0.23 off the line — more capture within 500 m
of a station, less beyond 1.5 km — but it nets out over the areas, because the walk-weighted
average the area skim already uses sits where the riders are. The busiest link is 1,644
against 1,673.

**The regression check** does not close to 1 %: with every TAZ pair given its area pair's
skims the TAZ run returns 3,970 underground (0.93), 3,050 ground (0.95), 4,668 design (0.92).
The trips drawn from the bus reproduce the area numbers exactly (3,841 / 2,972 / 4,481); the
trips induced from the car fall from 414 / 234 / 614 to 130 / 77 / 187. That is the pivot,
not the construction: the incremental logit's shift is proportional to S·(1 − S), and the TAZ
pairs' observed shares — allocations of survey trips to TAZs — are far more dispersed than
the area pair's, so the EB-smoothed shares (k = 20) give less room to shift. The induced
component is therefore a property of the level the pivot is run at, and the area level's
414 is its upper reading.

**Outputs** (`Output/skims/taz/`; the `area_plus_walk` alternative under `bus_area_plus_walk/`).
`lrt_capture_scenarios_taz.csv` (scenario × case, step 31's columns), `lrt_trips_2022_taz_{sc}_central.csv`
and `lrt_share_of_transit_taz_{sc}_central.csv` (174 × 174), `transit_share_2022_pivot_taz.csv`,
`taz_access_table.csv`, `lrt_trips_2022_area_from_taz_{sc}_central.csv` and `lrt_trips_by_area_taz_vs_area.csv`
(back to the areas), `taz_vs_area_capture_comparison.csv`, `trunk_link_flows_taz.csv`,
`lrt_boardings_by_station_area_taz.csv`, `lrt_boardings_by_station_taz.csv`, `lrt_capture_by_access_band.csv`,
`regression_check_area_skims_on_taz.csv`; figure `lrt_capture_taz_vs_area.png`.

**Limits.** The LRT's in-vehicle time, wait and feeder composition are the area pair's (the
TAZ's nearest station may differ from the area's representative one); only the two side walks
are at TAZ level, on the straight line × 1.3 from the polygon centroid (task C2's walking
network would replace both). The bus side is step 44's single best path (caveat 26) — dearer
than the area skim by construction, which is what moves the capture here; the alternative
construction shows the part of the change that is the walk alone. The pivot's prior for thin
TAZ pairs is the area pair's observed share; the 2022 TAZ layers are themselves an allocation
of survey trips to TAZs (purpose-blind, §8), so TAZ-pair shares carry that allocation's noise
through the EB smoothing. The scenarios and λ / premium cases are step 31's; the design-hour
and forecast steps are not rerun on the TAZ result.